In [1]:
SYMBOL = "BTCUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "rf"

In [2]:
# Parameters
SYMBOL = "ETHUSDT"
TARGET_HORIZON = 5
MODEL_TYPE = "xgb"


In [3]:
import os
import time
import json
import joblib
import pandas as pd
import numpy as np
import optuna
from functools import partial
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from features import add_features
from constants import DATA_DIR, MODEL_DIR
from utils import time_split, information_coefficient, rank_information_coefficient
from models import OBJECTIVES, MODEL_REGISTRY

/home/rachmiel/quant/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_DIR = os.path.join(MODEL_DIR, MODEL_TYPE)
PARQUET_PATH = f"{DATA_DIR}/{SYMBOL}_1m.parquet"

os.makedirs(MODEL_DIR, exist_ok=True)

In [5]:
model_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_model.joblib")
features_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_cols.json")
meta_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_meta.json")
fi_path = os.path.join(MODEL_DIR, f"{SYMBOL}__h{TARGET_HORIZON}_feature_importance.csv")
pred_path = os.path.join(MODEL_DIR, f"{SYMBOL}__{TARGET_HORIZON}_predictions.csv")

In [6]:
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] raw rows: {len(df):,}")

# add features + target
df, feature_cols = add_features(df, TARGET_HORIZON)

[info] raw rows: 284,679


In [7]:
df.head()

,open_time,open,high,low,close,volume,close_time,quote_asset_volume,num_trades,taker_buy_base_asset_volume,...,imbalance,imbalance_5,imbalance_15,trend_strength,vol_regime_ratio,is_trending,is_high_vol,mom_x_imb,mr_x_vol,trend_x_imb
0,2025-09-01 00:00:00+00:00,4391.83,4391.83,4386.25,4389.95,325.8367,2025-09-01 00:00:59.999999+00:00,1.429921e+06,3320,111.1554,...,-0.317723,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
1,2025-09-01 00:01:00+00:00,4389.96,4391.40,4389.68,4391.16,158.5513,2025-09-01 00:01:59.999999+00:00,6.961133e+05,1908,95.8326,...,0.208853,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
2,2025-09-01 00:02:00+00:00,4391.16,4391.16,4386.14,4388.19,187.0756,2025-09-01 00:02:59.999999+00:00,8.207380e+05,3039,100.6024,...,0.075527,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
3,2025-09-01 00:03:00+00:00,4388.19,4389.97,4386.33,4386.45,341.8429,2025-09-01 00:03:59.999999+00:00,1.500188e+06,2817,177.2970,...,0.037301,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN
4,2025-09-01 00:04:00+00:00,4386.45,4386.45,4375.39,4376.57,622.3295,2025-09-01 00:04:59.999999+00:00,2.725730e+06,5777,183.8020,...,-0.409310,-0.08107,NaN,NaN,NaN,0,0,NaN,NaN,NaN


In [8]:
target_col = f"target_ret_fwd_{TARGET_HORIZON}"

model_df = df[["open_time"] + feature_cols + [target_col]].copy()

# Remove:
# early rows where rolling features don’t exist yet
# rows where z-scores / ratios blew up
# rows where target is NaN (due to future shift)
model_df = model_df.replace([np.inf, -np.inf], np.nan)
model_df = model_df.dropna(subset=feature_cols + [target_col])

print(f"[info] usable rows after features: {len(model_df):,}")

train_df, test_df = time_split(model_df, train_frac=0.8)

# Further split the training set into train/valid for Optuna
optuna_train_df, valid_df = time_split(train_df, train_frac=0.8)

X_train = optuna_train_df[feature_cols]
y_train = optuna_train_df[target_col]

X_valid = valid_df[feature_cols]
y_valid = valid_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

train_start_time = pd.to_datetime(train_df["open_time"].iloc[0], utc=True)
train_end_time = pd.to_datetime(train_df["open_time"].iloc[-1], utc=True)

val_start_time = pd.to_datetime(valid_df["open_time"].iloc[0], utc=True)
val_end_time = pd.to_datetime(valid_df["open_time"].iloc[-1], utc=True)

test_start_time = pd.to_datetime(test_df["open_time"].iloc[0], utc=True)
test_end_time = pd.to_datetime(test_df["open_time"].iloc[-1], utc=True)

print(f"[info] optuna train rows: {len(optuna_train_df):,}")
print(f"[info] valid rows:        {len(valid_df):,}")
print(f"[info] test rows:         {len(test_df):,}")

[info] usable rows after features: 284,601
[info] optuna train rows: 182,144
[info] valid rows:        45,536
[info] test rows:         56,921


In [9]:
study = optuna.create_study(direction="maximize")
objective_fn = partial(
    OBJECTIVES[MODEL_TYPE],
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
)

study.optimize(objective_fn, n_trials=50, show_progress_bar=True)

print("\n[optuna] best trial")
print(f"value: {study.best_value:.6f}")
print("params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-18 12:24:06,466] A new study created in memory with name: no-name-33211926-eb36-43e4-a36b-d99eb062972f


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:02<?, ?it/s]

Best trial: 0. Best value: 0.0153257:   0%|          | 0/50 [00:02<?, ?it/s]

Best trial: 0. Best value: 0.0153257:   2%|▏         | 1/50 [00:02<02:12,  2.70s/it]

...

Best trial: 34. Best value: 0.0396667:  98%|█████████▊| 49/50 [04:01<00:08,  8.85s/it]

Best trial: 34. Best value: 0.0396667:  98%|█████████▊| 49/50 [04:01<00:08,  8.85s/it]

Best trial: 34. Best value: 0.0396667: 100%|██████████| 50/50 [04:01<00:00,  7.48s/it]

Best trial: 34. Best value: 0.0396667: 100%|██████████| 50/50 [04:01<00:00,  4.82s/it]

[I 2026-03-18 12:28:07,645] Trial 49 finished with value: 0.01837036257209552 and parameters: {'n_estimators': 1000, 'max_depth': 5, 'learning_rate': 0.0045022481294123215, 'subsample': 0.6264872036300346, 'colsample_bytree': 0.8660996583462676, 'min_child_weight': 19, 'reg_alpha': 0.0013336502625659905, 'reg_lambda': 0.22212876474971519}. Best is trial 34 with value: 0.039666712897303176.

[optuna] best trial
value: 0.039667
params:
  n_estimators: 600
  max_depth: 6
  learning_rate: 0.001026648157097117
  subsample: 0.9210972865886067
  colsample_bytree: 0.9648797662055696
  min_child_weight: 8
  reg_alpha: 9.768423138332662e-08
  reg_lambda: 1.93537108636511


In [10]:
best_params = study.best_params.copy()
best_params["random_state"] = 42
best_params["n_jobs"] = -1

X_train_full = train_df[feature_cols]
y_train_full = train_df[target_col]

final_model = MODEL_REGISTRY[MODEL_TYPE](**best_params)

start = time.time()
print(f"[training] fitting final {MODEL_TYPE}...")
final_model.fit(X_train_full, y_train_full)
print(f"[training] done in {time.time() - start:.2f}s")

[training] fitting final xgb...


[training] done in 2.72s


In [11]:
train_pred = final_model.predict(X_train_full)
test_pred = final_model.predict(X_test)

In [12]:
# evaluate
print("[eval] computing metrics...")
train_ic = information_coefficient(y_train_full.values, train_pred)
test_ic = information_coefficient(y_test.values, test_pred)

train_rank_ic = rank_information_coefficient(y_train_full.values, train_pred)
test_rank_ic = rank_information_coefficient(y_test.values, test_pred)

train_rmse = root_mean_squared_error(y_train_full, train_pred)
test_rmse = root_mean_squared_error(y_test, test_pred)

print("\n===== RESULTS =====")
print(f"Train IC:      {train_ic:.6f}")
print(f"Test IC:       {test_ic:.6f}")
print(f"Train Rank IC: {train_rank_ic:.6f}")
print(f"Test Rank IC:  {test_rank_ic:.6f}")
print(f"Train RMSE:    {train_rmse:.6f}")
print(f"Test RMSE:     {test_rmse:.6f}")

[eval] computing metrics...



===== RESULTS =====
Train IC:      0.271418
Test IC:       -0.017949
Train Rank IC: 0.079865
Test Rank IC:  0.004313
Train RMSE:    0.002112
Test RMSE:     0.002297


In [13]:
# feature importance
importances = pd.Series(
    final_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(importances)


===== FEATURE IMPORTANCE =====
imbalance_5         0.073138
vol_regime_ratio    0.062507
trend_x_imb         0.057110
volume_mom_5        0.056448
is_trending         0.054953
dist_ma_5           0.051724
range_ratio         0.043104
volume_z            0.042738
imbalance_15        0.036442
mom_5               0.034910
trades_z            0.034530
dist_ma_15_z        0.030831
vol_ratio_5_30      0.030129
mom_3               0.030124
trend_strength      0.029088
bar_range           0.029033
vol_15              0.029021
mr_x_vol            0.027834
mom_10              0.027120
num_trades_mom_5    0.025584
dist_ma_30          0.025348
vol_30              0.025074
mom_x_imb           0.021416
imbalance           0.021320
vol_5               0.019605
mom_15              0.018142
range_5             0.016910
range_15            0.015827
is_high_vol         0.015176
dist_ma_15          0.014814
dtype: float32


In [14]:
# save predictions
out = test_df[["open_time", target_col]].copy()
out["prediction"] = test_pred
out.to_csv(pred_path, index=False)
print(f"\n[saved] predictions -> {pred_path}")


[saved] predictions -> models/xgb/ETHUSDT__5_predictions.csv


In [15]:
# save model
joblib.dump(final_model, model_path)

# save feature columns
with open(features_path, "w") as f:
    json.dump(feature_cols, f, indent=2)

# save feature importance
importances.to_csv(fi_path, header=["importance"])

# save metadata
meta = {
    "symbol": SYMBOL,
    "target_horizon": int(TARGET_HORIZON),
    "target_col": target_col,
    "model_type": MODEL_TYPE,
    "study_best_value": float(study.best_value),
    "model_params": best_params,
    "n_features": int(len(feature_cols)),
    "feature_cols_path": str(features_path),
    "model_path": str(model_path),
    "feature_importance_path": str(fi_path) if fi_path is not None else None,
    "train_ic": train_ic,
    "test_ic": test_ic,
    "train_rank_ic": train_rank_ic,
    "test_rank_ic": test_rank_ic,
    "train_rmse": train_rmse,
    "test_rmse": test_rmse,
    "train_start_time": pd.Timestamp(train_start_time).isoformat(),
    "train_end_time": pd.Timestamp(train_end_time).isoformat(),
    "val_start_time": pd.Timestamp(val_start_time).isoformat(),
    "val_end_time": pd.Timestamp(val_end_time).isoformat(),
    "test_start_time": pd.Timestamp(test_start_time).isoformat(),
    "test_end_time": pd.Timestamp(test_end_time).isoformat()
}

with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print(f"[saved] model -> {model_path}")
print(f"[saved] features -> {features_path}")
print(f"[saved] feature importance -> {fi_path}")
print(f"[saved] metadata -> {meta_path}")

[saved] model -> models/xgb/ETHUSDT__h5_model.joblib
[saved] features -> models/xgb/ETHUSDT__h5_feature_cols.json
[saved] feature importance -> models/xgb/ETHUSDT__h5_feature_importance.csv
[saved] metadata -> models/xgb/ETHUSDT__h5_meta.json
